In [1]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from ultralytics import YOLO
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from torch.utils.data import DataLoader, TensorDataset

# ==============================
# SETTINGS
# ==============================

DATASET_ROOT = r"C:\Users\natra\OneDrive\Documents\cv project"
SEQ_LEN = 107

BATCH_SIZE = 32
EPOCHS = 100

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

yolo = YOLO("yolov5mu.pt")

# ==============================
# CBBA FEATURE EXTRACTION
# ==============================

def extract_cbba(video_path):

    cap = cv2.VideoCapture(video_path)

    areas = []

    while True:

        ret, frame = cap.read()
        if not ret:
            break

        results = yolo(frame, classes=[2], verbose=False)

        if len(results[0].boxes) == 0:
            areas.append(np.nan)
            continue

        boxes = results[0].boxes.xyxy.cpu().numpy()

        areas_boxes = (boxes[:,2]-boxes[:,0])*(boxes[:,3]-boxes[:,1])
        idx = np.argmax(areas_boxes)

        x1,y1,x2,y2 = boxes[idx]

        area = (x2-x1)*(y2-y1)

        areas.append(area)

    cap.release()

    areas = np.array(areas)

    valid = np.where(~np.isnan(areas))[0]

    if len(valid) < 10:
        return None

    start = valid[0]
    end = valid[-1]

    areas = areas[start:end+1]

    nans = np.isnan(areas)

    if np.any(nans):

        x = np.arange(len(areas))
        areas[nans] = np.interp(x[nans], x[~nans], areas[~nans])

    return areas


# ==============================
# RESIZE CBBA
# ==============================

def resize_cbba(cbba):

    x_old = np.linspace(0,1,len(cbba))
    x_new = np.linspace(0,1,SEQ_LEN)

    cbba = np.interp(x_new, x_old, cbba)

    return cbba


# ==============================
# DATASET CREATION
# ==============================

X = []
y = []

vehicle_names = []
max_cbba = 0

print("Extracting CBBA features...")

for vehicle in os.listdir(DATASET_ROOT):

    vehicle_path = os.path.join(DATASET_ROOT, vehicle)

    if not os.path.isdir(vehicle_path):
        continue

    for file in os.listdir(vehicle_path):

        if not file.endswith(".MP4"):
            continue

        video_path = os.path.join(vehicle_path,file)
        txt_path = video_path.replace(".MP4",".txt")

        if not os.path.exists(txt_path):
            continue

        with open(txt_path) as f:
            speed,_ = map(float,f.readline().split())

        cbba = extract_cbba(video_path)

        if cbba is None:
            continue

        cbba = resize_cbba(cbba)

        max_cbba = max(max_cbba, np.max(cbba))

        X.append(cbba)
        y.append(speed)
        vehicle_names.append(vehicle)

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)

# ==============================
# NORMALIZE
# ==============================

X = X / max_cbba

# ==============================
# MODEL (paper architecture)
# ==============================

class SpeedNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv1d(1,64,10),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64,64,10),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        self.fc = nn.Sequential(

            nn.Linear(64*20,64),
            nn.ReLU(),

            nn.Linear(64,1)
        )

    def forward(self,x):

        x = self.conv(x)

        x = x.view(x.size(0),-1)

        return self.fc(x)


model = SpeedNet().to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

criterion = nn.MSELoss()

# ==============================
# VEHICLE-WISE TRAINING
# ==============================

unique_vehicles = list(set(vehicle_names))

all_preds = []
all_true = []

for test_vehicle in unique_vehicles:

    print("Testing vehicle:", test_vehicle)

    train_idx = [i for i,v in enumerate(vehicle_names) if v != test_vehicle]
    test_idx = [i for i,v in enumerate(vehicle_names) if v == test_vehicle]

    X_train = torch.tensor(X[train_idx]).float().unsqueeze(1).to(DEVICE)
    y_train = torch.tensor(y[train_idx]).float().to(DEVICE)

    X_test = torch.tensor(X[test_idx]).float().unsqueeze(1).to(DEVICE)
    y_test = torch.tensor(y[test_idx]).float().to(DEVICE)

    train_loader = DataLoader(
        TensorDataset(X_train,y_train),
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    model.apply(lambda m: isinstance(m,nn.Conv1d) and m.reset_parameters())

    for epoch in range(EPOCHS):

        model.train()

        total_loss = 0

        for xb,yb in train_loader:

            optimizer.zero_grad()

            preds = model(xb).squeeze()

            loss = criterion(preds,yb)

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        if epoch % 10 == 0:
            print(f"Epoch {epoch} Loss {total_loss/len(train_loader):.2f}")

    model.eval()

    with torch.no_grad():

        preds = model(X_test).cpu().numpy().flatten()

    true = y_test.cpu().numpy()

    all_preds.extend(preds)
    all_true.extend(true)

# ==============================
# FINAL EVALUATION METRICS
# ==============================

rmse = np.sqrt(mean_squared_error(all_true,all_preds))
mae = mean_absolute_error(all_true,all_preds)
r2 = r2_score(all_true,all_preds)

print("\nEvaluation Metrics")
print("-------------------")
print("RMSE :", rmse)
print("MAE  :", mae)
print("R2 Score :", r2)

# ==============================
# SAVE MODEL
# ==============================

torch.save({
    "model":model.state_dict(),
    "max_cbba":max_cbba
},"speed_model_cbba.pth")

print("Model saved")

Device: cuda
Extracting CBBA features...
Dataset shape: (147, 107)
Testing vehicle: KiaSportage
Epoch 0 Loss 4974.93
Epoch 10 Loss 579.75
Epoch 20 Loss 456.49
Epoch 30 Loss 387.31
Epoch 40 Loss 307.87
Epoch 50 Loss 156.00
Epoch 60 Loss 120.59
Epoch 70 Loss 100.28
Epoch 80 Loss 85.96
Epoch 90 Loss 82.64
Testing vehicle: NissanQashqai
Epoch 0 Loss 4148.44
Epoch 10 Loss 385.12
Epoch 20 Loss 227.43
Epoch 30 Loss 126.01
Epoch 40 Loss 97.74
Epoch 50 Loss 88.43
Epoch 60 Loss 89.79
Epoch 70 Loss 79.46
Epoch 80 Loss 75.22
Epoch 90 Loss 73.77
Testing vehicle: MercedesAMG550
Epoch 0 Loss 3296.25
Epoch 10 Loss 359.81
Epoch 20 Loss 237.88
Epoch 30 Loss 156.91
Epoch 40 Loss 128.52
Epoch 50 Loss 112.30
Epoch 60 Loss 100.08
Epoch 70 Loss 93.95
Epoch 80 Loss 83.34
Epoch 90 Loss 77.97
Testing vehicle: Mazda3
Epoch 0 Loss 3399.45
Epoch 10 Loss 301.11
Epoch 20 Loss 141.70
Epoch 30 Loss 100.00
Epoch 40 Loss 91.91
Epoch 50 Loss 92.16
Epoch 60 Loss 87.42
Epoch 70 Loss 86.24
Epoch 80 Loss 83.95
Epoch 90 Loss 

In [ ]:
#1d cnn 

In [13]:
import cv2
import numpy as np
import torch
import torch.nn as nn
from ultralytics import YOLO

# ==============================
# SETTINGS
# ==============================

VIDEO_PATH = r"C:\Users\natra\Downloads\OpelInsignia\OpelInsignia\OpelInsignia_70.MP4"

SEQ_LEN = 107
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

# ==============================
# MODEL ARCHITECTURE
# ==============================

class SpeedNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv1d(1,64,10),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64,64,10),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        self.fc = nn.Sequential(

            nn.Linear(64*20,64),
            nn.ReLU(),

            nn.Linear(64,1)
        )

    def forward(self,x):

        x = self.conv(x)

        x = x.view(x.size(0),-1)

        return self.fc(x)

# ==============================
# LOAD MODEL
# ==============================

checkpoint = torch.load("speed_model_cbba.pth", map_location=DEVICE)

model = SpeedNet().to(DEVICE)
model.load_state_dict(checkpoint["model"])
model.eval()

max_cbba = checkpoint["max_cbba"]

print("Model loaded successfully")

# ==============================
# YOLO MODEL
# ==============================

yolo = YOLO("yolov5mu.pt")

# Detect car, bike, bus, truck
VEHICLE_CLASSES = [2,3,5,7]

# ==============================
# CBBA EXTRACTION
# ==============================

def extract_cbba(video_path):

    cap = cv2.VideoCapture(video_path)

    areas = []

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        results = yolo(frame, classes=VEHICLE_CLASSES, verbose=False)

        if len(results[0].boxes) == 0:

            areas.append(np.nan)
            continue

        boxes = results[0].boxes

        xyxy = boxes.xyxy.cpu().numpy()
        conf = boxes.conf.cpu().numpy()

        idx = np.argmax(conf)

        x1,y1,x2,y2 = xyxy[idx]

        area = (x2-x1)*(y2-y1)

        areas.append(area)

    cap.release()

    areas = np.array(areas)

    valid = np.where(~np.isnan(areas))[0]

    if len(valid) < 10:
        return None

    start = valid[0]
    end = valid[-1]

    areas = areas[start:end+1]

    nans = np.isnan(areas)

    if np.any(nans):

        x = np.arange(len(areas))
        areas[nans] = np.interp(x[nans], x[~nans], areas[~nans])

    return areas

# ==============================
# RESIZE CBBA
# ==============================

def resize_cbba(cbba):

    x_old = np.linspace(0,1,len(cbba))
    x_new = np.linspace(0,1,SEQ_LEN)

    cbba = np.interp(x_new, x_old, cbba)

    return cbba

# ==============================
# SPEED PREDICTION
# ==============================

def predict_speed(video_path):

    cbba = extract_cbba(video_path)

    if cbba is None:

        print("Vehicle not detected properly")
        return None

    cbba = resize_cbba(cbba)

    cbba = cbba / max_cbba

    tensor = torch.tensor(cbba).float().unsqueeze(0).unsqueeze(0).to(DEVICE)

    with torch.no_grad():

        speed = model(tensor).item()

    return speed

# ==============================
# RUN PREDICTION
# ==============================

speed = predict_speed(VIDEO_PATH)

if speed is not None:

    print("Predicted Speed:", round(speed,2), "km/h")

# ==============================
# DISPLAY VIDEO
# ==============================

cap = cv2.VideoCapture(VIDEO_PATH)

cv2.namedWindow("Vehicle Speed Estimation", cv2.WINDOW_NORMAL)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    results = yolo(frame, classes=VEHICLE_CLASSES, verbose=False)

    if len(results[0].boxes) > 0:

        boxes = results[0].boxes

        xyxy = boxes.xyxy.cpu().numpy()
        conf = boxes.conf.cpu().numpy()

        idx = np.argmax(conf)

        x1,y1,x2,y2 = xyxy[idx].astype(int)

        cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)

        if speed is not None:

            cv2.putText(
                frame,
                f"{speed:.2f} km/h",
                (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                (0,0,255),
                2
            )

    else:

        cv2.putText(
            frame,
            "0 km/h",
            (50,50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0,0,255),
            2
        )

    # Automatically scale video to fit screen
    h, w = frame.shape[:2]

    if w > 1280:
        frame = cv2.resize(frame,(1280, int(h*1280/w)))

    cv2.imshow("Vehicle Speed Estimation", frame)

    if cv2.waitKey(25) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

Device: cuda
Model loaded successfully
Predicted Speed: 84.59 km/h


In [17]:
import cv2
import numpy as np
import torch
import torch.nn as nn
from ultralytics import YOLO

# ==============================
# SETTINGS
# ==============================

VIDEO_PATH = r"C:\Users\natra\Downloads\OpelInsignia\OpelInsignia\OpelInsignia_35.MP4"

SEQ_LEN = 107
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

# ==============================
# FIX MOBILE VIDEO ORIENTATION
# ==============================

def fix_orientation(frame):

    h, w = frame.shape[:2]

    # If portrait video stored sideways
    if h > w * 1.3:
        frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)

    return frame


# ==============================
# MODEL ARCHITECTURE
# ==============================

class SpeedNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv1d(1,64,10),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64,64,10),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        self.fc = nn.Sequential(

            nn.Linear(64*20,64),
            nn.ReLU(),

            nn.Linear(64,1)
        )

    def forward(self,x):

        x = self.conv(x)
        x = x.view(x.size(0),-1)

        return self.fc(x)


# ==============================
# LOAD MODEL
# ==============================

checkpoint = torch.load("speed_model_cbba.pth", map_location=DEVICE)

model = SpeedNet().to(DEVICE)
model.load_state_dict(checkpoint["model"])
model.eval()

max_cbba = checkpoint["max_cbba"]

print("Model loaded successfully")


# ==============================
# YOLO MODEL
# ==============================

yolo = YOLO("yolov5mu.pt")

VEHICLE_CLASSES = [2,3,5,7]  # car, motorcycle, bus, truck


# ==============================
# CBBA EXTRACTION
# ==============================

def extract_cbba(video_path):

    cap = cv2.VideoCapture(video_path)

    areas = []

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame = fix_orientation(frame)

        results = yolo(frame, classes=VEHICLE_CLASSES, verbose=False)

        if len(results[0].boxes) == 0:

            areas.append(np.nan)
            continue

        boxes = results[0].boxes

        xyxy = boxes.xyxy.cpu().numpy()
        conf = boxes.conf.cpu().numpy()

        idx = np.argmax(conf)

        x1,y1,x2,y2 = xyxy[idx]

        area = (x2-x1)*(y2-y1)

        areas.append(area)

    cap.release()

    areas = np.array(areas)

    valid = np.where(~np.isnan(areas))[0]

    if len(valid) < 10:
        return None

    start = valid[0]
    end = valid[-1]

    areas = areas[start:end+1]

    nans = np.isnan(areas)

    if np.any(nans):

        x = np.arange(len(areas))
        areas[nans] = np.interp(x[nans], x[~nans], areas[~nans])

    return areas


# ==============================
# RESIZE CBBA
# ==============================

def resize_cbba(cbba):

    x_old = np.linspace(0,1,len(cbba))
    x_new = np.linspace(0,1,SEQ_LEN)

    cbba = np.interp(x_new, x_old, cbba)

    return cbba


# ==============================
# SPEED PREDICTION
# ==============================

def predict_speed(video_path):

    cbba = extract_cbba(video_path)

    if cbba is None:

        print("Vehicle not detected properly")
        return None

    cbba = resize_cbba(cbba)

    cbba = cbba / max_cbba

    tensor = torch.tensor(cbba).float().unsqueeze(0).unsqueeze(0).to(DEVICE)

    with torch.no_grad():

        speed = model(tensor).item()

    return speed


# ==============================
# RUN PREDICTION
# ==============================

speed = predict_speed(VIDEO_PATH)

if speed is not None:

    print("Predicted Speed:", round(speed,2), "km/h")


# ==============================
# DISPLAY VIDEO
# ==============================

cap = cv2.VideoCapture(VIDEO_PATH)

cv2.namedWindow("Vehicle Speed Estimation", cv2.WINDOW_NORMAL)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame = fix_orientation(frame)

    results = yolo(frame, classes=VEHICLE_CLASSES, verbose=False)

    if len(results[0].boxes) > 0:

        boxes = results[0].boxes

        xyxy = boxes.xyxy.cpu().numpy()
        conf = boxes.conf.cpu().numpy()

        idx = np.argmax(conf)

        x1,y1,x2,y2 = xyxy[idx].astype(int)

        cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)

        if speed is not None:

            cv2.putText(
                frame,
                f"{speed:.2f} km/h",
                (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                (0,0,255),
                2
            )

    else:

        cv2.putText(
            frame,
            "0 km/h",
            (50,50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0,0,255),
            2
        )


    # ==============================
    # AUTO SCREEN SCALING
    # ==============================

    screen_w = 1280
    screen_h = 720

    h, w = frame.shape[:2]

    scale = min(screen_w / w, screen_h / h)

    new_w = int(w * scale)
    new_h = int(h * scale)

    display = cv2.resize(frame,(new_w,new_h))

    cv2.imshow("Vehicle Speed Estimation", display)

    if cv2.waitKey(25) & 0xFF == 27:
        break


cap.release()
cv2.destroyAllWindows()

Device: cuda
Model loaded successfully
Predicted Speed: 37.88 km/h


In [3]:
#multiple vehicle 
import cv2
import numpy as np
import torch
import torch.nn as nn
from ultralytics import YOLO
from scipy.signal import medfilt

VIDEO_PATH = r"C:\Users\natra\Downloads\lowl ight test.mp4"
SEQ_LEN = 107
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

# ==============================
# MODEL
# ==============================

class SpeedNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(1,64,10),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64,64,10),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        self.fc = nn.Sequential(
            nn.Linear(64*20,64),
            nn.ReLU(),
            nn.Linear(64,1)
        )

    def forward(self,x):

        x = self.conv(x)
        x = x.view(x.size(0),-1)

        return self.fc(x)


# ==============================
# LOAD MODEL
# ==============================

checkpoint = torch.load("speed_model_cbba.pth", map_location=DEVICE)

model = SpeedNet().to(DEVICE)
model.load_state_dict(checkpoint["model"])
model.eval()

max_cbba = checkpoint["max_cbba"]

print("Model loaded successfully")

# ==============================
# YOLO DETECTOR
# ==============================

yolo = YOLO("yolov5mu.pt")

VEHICLE_CLASSES = [2,3,5,7]

# ==============================
# TRACKING STRUCTURES
# ==============================

tracks = {}
centers_prev = {}
speeds = {}
next_id = 0


def assign_id(center):

    global next_id

    for vid,prev in centers_prev.items():

        dist = np.linalg.norm(np.array(center)-np.array(prev))

        if dist < 60:
            centers_prev[vid] = center
            return vid

    next_id += 1
    centers_prev[next_id] = center

    return next_id


# ==============================
# RESIZE CBBA
# ==============================

def resize_cbba(cbba):

    x_old = np.linspace(0,1,len(cbba))
    x_new = np.linspace(0,1,SEQ_LEN)

    return np.interp(x_new,x_old,cbba)


# ==============================
# VIDEO
# ==============================

cap = cv2.VideoCapture(VIDEO_PATH)

cv2.namedWindow("Vehicle Speed Estimation",cv2.WINDOW_NORMAL)

while True:

    ret,frame = cap.read()

    if not ret:
        break

    results = yolo(frame,classes=VEHICLE_CLASSES,verbose=False)

    if len(results[0].boxes)>0:

        boxes = results[0].boxes.xyxy.cpu().numpy()

        for box in boxes:

            x1,y1,x2,y2 = map(int,box)

            center = ((x1+x2)//2,(y1+y2)//2)

            vid = assign_id(center)

            area = (x2-x1)*(y2-y1)

            if vid not in tracks:
                tracks[vid] = []

            tracks[vid].append(area)

            # keep only latest frames
            if len(tracks[vid]) > SEQ_LEN:
                tracks[vid] = tracks[vid][-SEQ_LEN:]

            # Predict only once after full sequence
            if len(tracks[vid]) == SEQ_LEN and vid not in speeds:

                areas = np.array(tracks[vid])

                # ===== Noise removal =====
                areas = medfilt(areas, kernel_size=7)

                # ===== Signal smoothing =====
                kernel = np.ones(5)/5
                areas = np.convolve(areas,kernel,mode='same')

                # ===== Handle missing values =====
                valid = np.where(~np.isnan(areas))[0]

                if len(valid) < 10:
                    continue

                start = valid[0]
                end = valid[-1]

                areas = areas[start:end+1]

                nans = np.isnan(areas)

                if np.any(nans):

                    x = np.arange(len(areas))
                    areas[nans] = np.interp(x[nans],x[~nans],areas[~nans])

                # ===== Resize to training length =====
                cbba = resize_cbba(areas)

                # ===== Normalize =====
                cbba = cbba / max_cbba

                tensor = torch.tensor(cbba).float().unsqueeze(0).unsqueeze(0).to(DEVICE)

                with torch.no_grad():

                    raw_speed = model(tensor).item()

                # ===== Calibration Fix =====
                speed = 0.88 * raw_speed - 2

                speeds[vid] = speed

            # draw bounding box
            cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)

            if vid in speeds:

                cv2.putText(
                    frame,
                    f"ID {vid}: {speeds[vid]:.2f} km/h",
                    (x1,y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0,0,255),
                    2
                )

    cv2.imshow("Vehicle Speed Estimation",frame)

    if cv2.waitKey(25) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

Device: cuda
Model loaded successfully


In [25]:
#saving the output video 
import cv2
import numpy as np
import torch
import torch.nn as nn
from ultralytics import YOLO
from scipy.signal import medfilt

VIDEO_PATH = r"C:\Users\natra\Downloads\truck_test_2.mp4"
OUTPUT_PATH = r"C:\Users\natra\Downloads\cv model outputs\trained model\truck_output_speed.mp4"

SEQ_LEN = 107
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

# ==============================
# MODEL
# ==============================

class SpeedNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(1,64,10),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64,64,10),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        self.fc = nn.Sequential(
            nn.Linear(64*20,64),
            nn.ReLU(),
            nn.Linear(64,1)
        )

    def forward(self,x):

        x = self.conv(x)
        x = x.view(x.size(0),-1)

        return self.fc(x)


# ==============================
# LOAD MODEL
# ==============================

checkpoint = torch.load("speed_model_cbba.pth", map_location=DEVICE)

model = SpeedNet().to(DEVICE)
model.load_state_dict(checkpoint["model"])
model.eval()

max_cbba = checkpoint["max_cbba"]

print("Model loaded successfully")

# ==============================
# YOLO DETECTOR
# ==============================

yolo = YOLO("yolov5mu.pt")

VEHICLE_CLASSES = [2,3,5,7]

# ==============================
# TRACKING STRUCTURES
# ==============================

tracks = {}
centers_prev = {}
speeds = {}
next_id = 0


def assign_id(center):

    global next_id

    for vid,prev in centers_prev.items():

        dist = np.linalg.norm(np.array(center)-np.array(prev))

        if dist < 60:
            centers_prev[vid] = center
            return vid

    next_id += 1
    centers_prev[next_id] = center

    return next_id


# ==============================
# RESIZE CBBA
# ==============================

def resize_cbba(cbba):

    x_old = np.linspace(0,1,len(cbba))
    x_new = np.linspace(0,1,SEQ_LEN)

    return np.interp(x_new,x_old,cbba)


# ==============================
# VIDEO
# ==============================

cap = cv2.VideoCapture(VIDEO_PATH)

# 👉 ADD VIDEO WRITER
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

cv2.namedWindow("Vehicle Speed Estimation",cv2.WINDOW_NORMAL)

while True:

    ret,frame = cap.read()

    if not ret:
        break

    results = yolo(frame,classes=VEHICLE_CLASSES,verbose=False)

    if len(results[0].boxes)>0:

        boxes = results[0].boxes.xyxy.cpu().numpy()

        for box in boxes:

            x1,y1,x2,y2 = map(int,box)

            center = ((x1+x2)//2,(y1+y2)//2)

            vid = assign_id(center)

            area = (x2-x1)*(y2-y1)

            if vid not in tracks:
                tracks[vid] = []

            tracks[vid].append(area)

            if len(tracks[vid]) > SEQ_LEN:
                tracks[vid] = tracks[vid][-SEQ_LEN:]

            if len(tracks[vid]) == SEQ_LEN and vid not in speeds:

                areas = np.array(tracks[vid])

                areas = medfilt(areas, kernel_size=7)

                kernel = np.ones(5)/5
                areas = np.convolve(areas,kernel,mode='same')

                valid = np.where(~np.isnan(areas))[0]

                if len(valid) < 10:
                    continue

                start = valid[0]
                end = valid[-1]

                areas = areas[start:end+1]

                nans = np.isnan(areas)

                if np.any(nans):

                    x = np.arange(len(areas))
                    areas[nans] = np.interp(x[nans],x[~nans],areas[~nans])

                cbba = resize_cbba(areas)

                cbba = cbba / max_cbba

                tensor = torch.tensor(cbba).float().unsqueeze(0).unsqueeze(0).to(DEVICE)

                with torch.no_grad():

                    raw_speed = model(tensor).item()

                speed = 0.88 * raw_speed - 2

                speeds[vid] = speed

            cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)

            if vid in speeds:

                cv2.putText(
                    frame,
                    f"ID {vid}: {speeds[vid]:.2f} km/h",
                    (x1,y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0,0,255),
                    2
                )

    # 👉 SAVE FRAME
    out.write(frame)

    cv2.imshow("Vehicle Speed Estimation",frame)

    if cv2.waitKey(25) & 0xFF == 27:
        break

cap.release()
out.release()  # 👉 IMPORTANT
cv2.destroyAllWindows()

print("Output video saved at:", OUTPUT_PATH)

Device: cuda
Model loaded successfully
Output video saved at: C:\Users\natra\Downloads\cv model outputs\trained model\truck_output_speed.mp4


In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn
from ultralytics import YOLO

# ==============================
# SETTINGS
# ==============================

VIDEO_PATH = r"C:\Users\natra\Downloads\truck_test_2.mp4"
OUTPUT_PATH = r"C:\Users\natra\Downloads\cv model outputs\trained model\truck_output_speed.mp4"

SEQ_LEN = 107
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

# ==============================
# FIX MOBILE VIDEO ORIENTATION
# ==============================

def fix_orientation(frame):

    h, w = frame.shape[:2]

    if h > w * 1.3:
        frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)

    return frame


# ==============================
# MODEL ARCHITECTURE
# ==============================

class SpeedNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv1d(1,64,10),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64,64,10),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        self.fc = nn.Sequential(

            nn.Linear(64*20,64),
            nn.ReLU(),

            nn.Linear(64,1)
        )

    def forward(self,x):

        x = self.conv(x)
        x = x.view(x.size(0),-1)

        return self.fc(x)


# ==============================
# LOAD MODEL
# ==============================

checkpoint = torch.load("speed_model_cbba.pth", map_location=DEVICE)

model = SpeedNet().to(DEVICE)
model.load_state_dict(checkpoint["model"])
model.eval()

max_cbba = checkpoint["max_cbba"]

print("Model loaded successfully")


# ==============================
# YOLO MODEL
# ==============================

yolo = YOLO("yolov5mu.pt")

VEHICLE_CLASSES = [2,3,5,7]


# ==============================
# CBBA EXTRACTION
# ==============================

def extract_cbba(video_path):

    cap=cv2.VideoCapture(video_path)

    areas=[]

    while True:

        ret,frame=cap.read()

        if not ret:
            break

        frame=fix_orientation(frame)

        results=yolo(frame,classes=VEHICLE_CLASSES,verbose=False)

        if len(results[0].boxes)==0:
            areas.append(np.nan)
            continue

        boxes=results[0].boxes

        xyxy=boxes.xyxy.cpu().numpy()
        conf=boxes.conf.cpu().numpy()

        idx=np.argmax(conf)

        x1,y1,x2,y2=xyxy[idx]

        area=(x2-x1)*(y2-y1)

        areas.append(area)

    cap.release()

    areas=np.array(areas)

    valid=np.where(~np.isnan(areas))[0]

    if len(valid)<10:
        return None

    start=valid[0]
    end=valid[-1]

    areas=areas[start:end+1]

    nans=np.isnan(areas)

    if np.any(nans):

        x=np.arange(len(areas))
        areas[nans]=np.interp(x[nans],x[~nans],areas[~nans])

    return areas


# ==============================
# RESIZE CBBA
# ==============================

def resize_cbba(cbba):

    x_old=np.linspace(0,1,len(cbba))
    x_new=np.linspace(0,1,SEQ_LEN)

    return np.interp(x_new,x_old,cbba)


# ==============================
# SPEED PREDICTION
# ==============================

def predict_speed(video_path):

    cbba=extract_cbba(video_path)

    if cbba is None:
        print("Vehicle not detected properly")
        return None

    cbba=resize_cbba(cbba)
    cbba=cbba/max_cbba

    tensor=torch.tensor(cbba).float().unsqueeze(0).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        speed=model(tensor).item()

    return speed


# ==============================
# RUN PREDICTION
# ==============================

speed=predict_speed(VIDEO_PATH)

if speed is not None:
    print("Predicted Speed:",round(speed,2),"km/h")


# ==============================
# DISPLAY + SAVE VIDEO
# ==============================

cap=cv2.VideoCapture(VIDEO_PATH)

fps=cap.get(cv2.CAP_PROP_FPS)
width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc=cv2.VideoWriter_fourcc(*"mp4v")
out=cv2.VideoWriter(OUTPUT_PATH,fourcc,fps,(width,height))

cv2.namedWindow("Vehicle Speed Estimation",cv2.WINDOW_NORMAL)

while True:

    ret,frame=cap.read()

    if not ret:
        break

    frame=fix_orientation(frame)

    results=yolo(frame,classes=VEHICLE_CLASSES,verbose=False)

    if len(results[0].boxes)>0:

        boxes=results[0].boxes

        xyxy=boxes.xyxy.cpu().numpy()
        conf=boxes.conf.cpu().numpy()

        idx=np.argmax(conf)

        x1,y1,x2,y2=xyxy[idx].astype(int)

        cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)

        if speed is not None:

            cv2.putText(
                frame,
                f"{speed:.2f} km/h",
                (x1,y1-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                (0,0,255),
                2
            )

    else:

        cv2.putText(
            frame,
            "0 km/h",
            (50,50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0,0,255),
            2
        )

    # SAVE FRAME
    out.write(frame)

    # DISPLAY (scaled)
    screen_w=1280
    screen_h=720

    h,w=frame.shape[:2]

    scale=min(screen_w/w,screen_h/h)

    new_w=int(w*scale)
    new_h=int(h*scale)

    display=cv2.resize(frame,(new_w,new_h))

    cv2.imshow("Vehicle Speed Estimation",display)

    if cv2.waitKey(25)&0xFF==27:
        break

cap.release()
out.release()
cv2.destroyAllWindows()

print("Output saved at:",OUTPUT_PATH)